<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/Day_15_Reranking_chunk_overlap_chunk_size_tradeoff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Why do we retrieve Top-k chunks?**
Think about:
What happens if only 1 chunk is retrieved?
What happens if 50 chunks are retrieved?
Why is a middle value usually better?

**We retrieve the top-k chunks because a single chunk may not contain all the information required to answer a question, while retrieving too many chunks introduces noise and increases token usage. Using a moderate value such as k=3 or k=5 provides a good balance between recall and precision, giving the LLM enough relevant context to generate accurate answers efficiently.**

How can a re-ranker improve the final answer quality?

FAISS retrieves chunks using vector similarity, but the most relevant chunk may not always be ranked first. A re-ranker uses a cross-encoder to evaluate the relationship between the question and each retrieved chunk more accurately. It assigns new relevance scores and reorders the chunks. As a result, the most relevant chunk can move from rank 8 to rank 1, allowing the LLM to receive better context and generate a more accurate answer.

What problem does overlap solve?

We use chunk overlap to preserve context between consecutive chunks. Without overlap, important information that lies at the boundary of two chunks may get split and lose its meaning. By repeating a small portion of text in the next chunk, we ensure that related information remains available during retrieval, which improves retrieval accuracy and answer quality.

In [ ]:
sentences = [
    "Machine Learning is a subset of AI",
    "Deep Learning uses neural networks",
    "Cricket is a popular sport",
    "Football is played worldwide"
]

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model=SentenceTransformer(
    'all-MiniLM-L6-v2'
)

In [ ]:
embeddings=model.encode(sentences)

In [ ]:
print(embeddings.shape)

Sentence 1 -> 384 numbers
Sentence 2 -> 384 numbers
Sentence 3 -> 384 numbers
Sentence 4 -> 384 numbers

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import numpy as np


In [ ]:
dimension=embeddings.shape[1]

In [ ]:
index=faiss.IndexFlatL2(dimension)

In [ ]:
index.add(
    np.array(embeddings).astype('float32')
)

In [ ]:
print(index.ntotal) #FAISS stored 4 vectors

In [ ]:
question='what is artificial intelligence?'

q_embeddings=model.encode([question])

distances,indicies=index.search(
    np.array(q_embeddings).astype('float32'),
    k=2
)
print(indicies)

In [ ]:
for idx in indicies[0]:
    print(sentences[idx])